In [38]:
!pip install torchmetrics

In [39]:
import pandas as pd
import torch
import sklearn
import numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
import torchmetrics
from torchmetrics import MetricCollection

In [40]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED)

In [41]:
df = pd.read_csv("/content/drive/MyDrive/colab ex/datasets/healthcare-dataset-stroke-data.csv")
df = df.drop('id', axis=1)

cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[cat_cols] = encoder.fit_transform(df[cat_cols])

df['bmi'] = df['bmi'].fillna(df['bmi'].median())

X = df.drop('stroke', axis=1).values
y = df['stroke'].values

train_X, temp_X, train_y, temp_y = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

val_X, test_X, val_y, test_y = train_test_split(
    temp_X, temp_y,
    test_size = 0.5,
    random_state=SEED,
    stratify=temp_y
)

scaler = RobustScaler()
train_X = scaler.fit_transform(train_X)
val_X = scaler.transform(val_X)
test_X = scaler.transform(test_X)

class StrokeDataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype=torch.float)
    self.y = torch.tensor(y, dtype=torch.float).unsqueeze(1)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

  def __len__(self):
    return len(self.y)

train_ds = StrokeDataset(train_X, train_y)
val_ds = StrokeDataset(val_X, val_y)
test_ds = StrokeDataset(test_X, test_y)

class_count = np.bincount(train_y)
class_weight = 1/class_count
sample_weight = class_weight[train_y]
sampler = WeightedRandomSampler(sample_weight, num_samples=len(sample_weight))

train_loader = DataLoader(train_ds, sampler=sampler, batch_size=32, num_workers=2, persistent_workers=True, pin_memory=True)
val_loader = DataLoader(val_ds, shuffle=False, batch_size=32, num_workers=2, persistent_workers=True, pin_memory=True)
test_loader = DataLoader(test_ds, shuffle=False, batch_size=32, num_workers=2, persistent_workers=True, pin_memory=True)

In [42]:
class_count

array([3889,  199])

In [89]:
class StrokeModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(

            nn.Linear(10, 48),
            nn.ReLU(),
            nn.BatchNorm1d(48),
            nn.Dropout(0.4),

            nn.Linear(48, 24),
            nn.ReLU(),
            nn.BatchNorm1d(24),
            nn.Dropout(0.4),

            nn.Linear(24, 12),
            nn.ReLU(),
            nn.BatchNorm1d(12),
            nn.Dropout(0.3),

            nn.Linear(12, 1)
    )

  def forward(self, x):
    return self.net(x)

model = StrokeModel().to(device)

In [90]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logit, target):
        bce_loss = self.bce(logit, target)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

loss_fn = FocalLoss(alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
schedular = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-3,
    epochs=80,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,
    final_div_factor=80
)
metrics = MetricCollection({
    "Precision":torchmetrics.classification.BinaryPrecision(),
    "Recall":torchmetrics.classification.BinaryRecall(),
    "F1":torchmetrics.classification.BinaryF1Score(),
    "ROC-AUC":torchmetrics.classification.BinaryAUROC(),
    "PR-AUC":torchmetrics.classification.BinaryAveragePrecision()
}).to(device)


In [91]:
def train_epoch(model, loader, loss_fn, device, optimizer):
  model.train()
  total_loss = 0
  for X, y in loader:
    X = X.to(device)
    y = y.to(device).squeeze(-1)
    optimizer.zero_grad()
    logit = model(X).squeeze(-1)
    loss = loss_fn(logit, y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item() * X.size(0)
  return total_loss / len(loader.dataset)

def eval_epoch(model, loader, device, metrics):
  model.eval()
  metrics.reset()
  with torch.no_grad():
    for X, y in loader:
      X = X.to(device)
      y = y.to(device).long()
      logit = model(X)
      metrics.update(torch.sigmoid(logit), y)
  return metrics.compute()

In [92]:
EPOCHS = 100
best_PR = -float('inf')
patience_counter = 0
patience = 12
delta = 0.001
best_model = None

for epoch in range(EPOCHS):
  train_loss = train_epoch(model, train_loader, loss_fn, device, optimizer)

  metrics_value = eval_epoch(model, val_loader, device, metrics)
  cur_PR = metrics_value["PR-AUC"].item()
  schedular.step(cur_PR)
  print(f"EPOCH[{epoch}] train_loss:{train_loss:.4f} | val_PR-AUC: {cur_PR:.4f} | Precision: {metrics_value['Precision']:.4f} | Recall: {metrics_value['Recall']:.4f} | F1: {metrics_value['F1']:.4f}")

  if cur_PR > best_PR + delta:
    patience_counter = 0
    best_PR = cur_PR
    best_model = model.state_dict().copy()
  else:
    patience_counter +=1
    if patience_counter >= patience:
      print("Early Stopping triggered")
      break


if best_model:
  model.load_state_dict(best_model)

test_metrics = eval_epoch(model, test_loader, device, metrics)
print("Test Result")
print(f"PR-AUC:{test_metrics["PR-AUC"]:.4f}")
print(f"Precision{test_metrics['Precision']:.4f}")
print(f"Recall:{test_metrics['Recall']:.4f}")
print(f"F1:{test_metrics['F1']:.4f}")

/tmp/ipython-input-423/2614191217.py:13: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  schedular.step(cur_PR)


EPOCH[0] train_loss:0.0579 | val_PR-AUC: 0.1527 | Precision: 0.1964 | Recall: 0.4400 | F1: 0.2716
EPOCH[1] train_loss:0.0490 | val_PR-AUC: 0.2796 | Precision: 0.1688 | Recall: 0.5200 | F1: 0.2549
EPOCH[2] train_loss:0.0460 | val_PR-AUC: 0.2770 | Precision: 0.1810 | Recall: 0.7600 | F1: 0.2923
EPOCH[3] train_loss:0.0430 | val_PR-AUC: 0.2967 | Precision: 0.1570 | Recall: 0.7600 | F1: 0.2603
EPOCH[4] train_loss:0.0408 | val_PR-AUC: 0.3140 | Precision: 0.1750 | Recall: 0.8400 | F1: 0.2897
EPOCH[5] train_loss:0.0392 | val_PR-AUC: 0.3089 | Precision: 0.1438 | Recall: 0.8800 | F1: 0.2472
EPOCH[6] train_loss:0.0388 | val_PR-AUC: 0.3149 | Precision: 0.1401 | Recall: 0.8800 | F1: 0.2418
EPOCH[7] train_loss:0.0385 | val_PR-AUC: 0.2979 | Precision: 0.1341 | Recall: 0.8800 | F1: 0.2328
EPOCH[8] train_loss:0.0380 | val_PR-AUC: 0.2897 | Precision: 0.1333 | Recall: 0.8800 | F1: 0.2316
EPOCH[9] train_loss:0.0364 | val_PR-AUC: 0.2894 | Precision: 0.1287 | Recall: 0.8800 | F1: 0.2245
EPOCH[10] train_loss